# Time-series model benchmark — bản đã sửa

Notebook này tự động tìm thư mục dự án/CSV, xử lý thư viện tùy chọn, chạy walk-forward không nhìn tương lai và xuất prediction đúng ngày mục tiêu.

Cài thư viện khi cần:
```bash
pip install numpy pandas tqdm statsmodels pmdarima arch neuralforecast torch chronos-forecasting
```


In [ ]:
# ============================================================
# 1) TIEN ICH CHUNG
# ============================================================
from __future__ import annotations

import os
import re
import time
import warnings
from math import erf, sqrt
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TARGET = "Gia_target"
DATE_COL = "date"
HORIZONS = [1, 5, 21, 63]
INITIAL_FRAC = 0.6
STEP = 5

FEATURES_ALL = [
    "target_lag1", "target_lag2", "target_lag3", "target_ret_lag1",
    "MA5", "MA10", "std5", "dayofweek", "month",
    "london_vnd_kg_lag1", "usdvnd_lag1", "diesel", "diesel_chg_1m",
    "diesel_chg_3m", "Luong_lag1m", "rain_90d", "oni", "waterbal_90d",
    "area_tn", "prod_tn", "yield_tn", "tonkho_tan", "dongia_lag1m",
    "dongia_ret_lag1m",
]


def _now():
    return time.strftime("%H:%M:%S")


def discover_project_root(explicit=None):
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    if os.getenv("COFPRED_ROOT"):
        candidates.append(Path(os.environ["COFPRED_ROOT"]).expanduser())
    candidates.append(Path.cwd())
    if os.name == "nt":
        candidates.append(Path(r"E:\FPT\AI\SEM8_AI\DAP391m\project\CofPred"))
    for base in candidates:
        for candidate in [base, *base.parents]:
            if (candidate / "data" / "processed" / "gia_cafe_master_full.csv").exists():
                return candidate.resolve()
    return Path.cwd().resolve()


PROJECT_ROOT = discover_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gia_cafe_master_full.csv"
RESULTS_ROOT = PROJECT_ROOT / "results"
PREDS_DIR = RESULTS_ROOT / "preds"
TS_DIR = RESULTS_ROOT / "TS_model"
NF_MODEL_DIR = RESULTS_ROOT / "saved_models" / "neuralforecast"
for folder in (PREDS_DIR, TS_DIR, NF_MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def save_preds(model_name, horizon, dates, y_true, y_pred, anchor=None, outdir=PREDS_DIR):
    dates = pd.to_datetime(list(dates), errors="coerce")
    y_true = np.asarray(list(y_true), float)
    y_pred = np.asarray(list(y_pred), float)
    if not (len(dates) == len(y_true) == len(y_pred)):
        raise ValueError(f"Do dai khong khop: dates={len(dates)}, true={len(y_true)}, pred={len(y_pred)}")
    frame = pd.DataFrame({
        "date": dates,
        "y_true": y_true,
        "y_pred": y_pred,
        "error": y_true - y_pred,
        "abs_error": np.abs(y_true - y_pred),
    })
    if anchor is not None:
        anchor = np.asarray(list(anchor), float)
        if len(anchor) != len(frame):
            raise ValueError("Do dai anchor khong khop")
        frame["anchor"] = anchor
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(model_name)).strip("_") or "model"
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / f"preds_{safe}_h{int(horizon)}.csv"
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    return path


try:
    from tqdm.auto import tqdm
    def progress(iterator, desc=""):
        return tqdm(list(iterator), desc=desc, leave=False)
except Exception:
    def progress(iterator, desc=""):
        items = list(iterator)
        for i, value in enumerate(items, 1):
            if i == 1 or i % 20 == 0 or i == len(items):
                print(f"[{_now()}] {desc}: {i}/{len(items)}", flush=True)
            yield value


def detect_date_column(columns):
    for name in ("date", "Date", "Ngay", "ngay", "DATE"):
        if name in columns:
            return name
    raise KeyError("Khong tim thay cot ngay: date/Date/Ngay/ngay/DATE")


def load_data(data_path=DATA_PATH):
    global DATE_COL
    data_path = Path(data_path)
    if not data_path.exists():
        raise FileNotFoundError(
            f"Khong tim thay {data_path}. Hay sua PROJECT_ROOT/DATA_PATH hoac dat COFPRED_ROOT."
        )
    df = pd.read_csv(data_path)
    DATE_COL = detect_date_column(df.columns)
    if TARGET not in df.columns:
        raise KeyError(f"Khong co cot target {TARGET}")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    df = (df.dropna(subset=[DATE_COL])
            .sort_values(DATE_COL)
            .drop_duplicates(DATE_COL, keep="last")
            .reset_index(drop=True))
    features = [c for c in FEATURES_ALL if c in df.columns]
    if features:
        df[features] = df[features].apply(pd.to_numeric, errors="coerce")
    df["prev_price"] = df[TARGET].shift(1)
    before = len(df)
    df = df.dropna(subset=[TARGET, "prev_price"] + features).reset_index(drop=True)
    if len(df) < 100:
        raise ValueError(f"Chi con {len(df)} dong hop le; du lieu qua it.")
    print(f"[load] {data_path}")
    print(f"[load] giu {len(df)}/{before} dong, date_col={DATE_COL}, features={len(features)}")
    return df


def evaluation_origins(n, horizon, initial=INITIAL_FRAC, step=STEP):
    start = int(n * initial)
    return sorted(range(n - 1 - horizon, start - 1, -step))


def mae(y, yhat):
    return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(yhat, float))))


def rmse(y, yhat):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(yhat, float)) ** 2)))


def directional_accuracy(base, y_true, y_pred):
    return float(np.mean(
        np.sign(np.asarray(y_true, float) - np.asarray(base, float)) ==
        np.sign(np.asarray(y_pred, float) - np.asarray(base, float))
    ))


def diebold_mariano(e_model, e_naive, h=1, loss="mae"):
    e1, e2 = np.asarray(e_model, float), np.asarray(e_naive, float)
    valid = np.isfinite(e1) & np.isfinite(e2)
    e1, e2 = e1[valid], e2[valid]
    d = np.abs(e1) - np.abs(e2) if loss == "mae" else e1 ** 2 - e2 ** 2
    n = len(d)
    if n < 2:
        return float("nan"), float("nan")
    var = float(np.var(d, ddof=0))
    for k in range(1, min(h, n)):
        if n - k > 1:
            var += 2 * (1 - k / h) * float(np.cov(d[k:], d[:-k], ddof=0)[0, 1])
    if not np.isfinite(var) or var <= 0:
        return float("nan"), float("nan")
    dm = float(d.mean() / sqrt(var / n))
    p = float(2 * (1 - 0.5 * (1 + erf(abs(dm) / sqrt(2)))))
    return dm, p


def walk_forward_eval(serie, forecast_fn, horizon, exog=None, dates=None, desc="walk-forward"):
    serie = np.asarray(serie, float)
    origins = evaluation_origins(len(serie), horizon)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(origins, desc):
        train_y = serie[:d + 1]
        train_exog = exog[:d + 1] if exog is not None else None
        future_exog = np.tile(exog[d:d + 1], (horizon, 1)) if exog is not None else None
        try:
            prediction = float(forecast_fn(train_y, train_exog, future_exog, horizon))
        except Exception:
            prediction = float(train_y[-1])
        y_pred.append(prediction)
        y_true.append(float(serie[d + horizon]))
        y_naive.append(float(serie[d]))
        base.append(float(serie[d]))
    result = {k: np.asarray(v, float) for k, v in {
        "y_true": y_true, "y_pred": y_pred, "y_naive": y_naive, "base": base
    }.items()}
    result["origins"] = np.asarray(origins, int)
    if dates is not None:
        result["target_dates"] = pd.to_datetime(np.asarray(dates)[result["origins"] + horizon])
    return result


def summarize(tag, horizon, result):
    e_model = result["y_true"] - result["y_pred"]
    e_naive = result["y_true"] - result["y_naive"]
    dm, p = diebold_mariano(e_model, e_naive, h=horizon, loss="mae")
    naive_mae = mae(result["y_true"], result["y_naive"])
    model_mae = mae(result["y_true"], result["y_pred"])
    return {
        "model": tag, "h": horizon, "n": len(result["y_true"]),
        "MAE": round(model_mae, 1), "MAE_naive": round(naive_mae, 1),
        "dMAE_pct": round(100 * (model_mae / naive_mae - 1), 2) if naive_mae > 0 else np.nan,
        "RMSE": round(rmse(result["y_true"], result["y_pred"]), 1),
        "DA": round(directional_accuracy(result["base"], result["y_true"], result["y_pred"]), 3),
        "DM": round(dm, 3) if np.isfinite(dm) else np.nan,
        "DM_p": round(p, 4) if np.isfinite(p) else np.nan,
    }


## 2. Hàm dự báo từng model

In [ ]:
# ============================================================
# 2) HAM DU BAO
# ============================================================
def fc_naive(train_y, train_exog, future_exog, horizon):
    return float(train_y[-1])


def fc_autoarima(train_y, train_exog, future_exog, horizon):
    try:
        import pmdarima as pm
    except ImportError as exc:
        raise RuntimeError("Thieu pmdarima: pip install pmdarima") from exc
    model = pm.auto_arima(
        train_y, seasonal=False, d=None, max_p=5, max_q=5,
        suppress_warnings=True, error_action="ignore", stepwise=True,
    )
    return float(np.asarray(model.predict(n_periods=horizon))[-1])


def fc_sarima(train_y, train_exog, future_exog, horizon):
    try:
        import pmdarima as pm
    except ImportError as exc:
        raise RuntimeError("Thieu pmdarima: pip install pmdarima") from exc
    model = pm.auto_arima(
        train_y, seasonal=True, m=5, d=None, D=None,
        max_p=3, max_q=3, max_P=2, max_Q=2,
        suppress_warnings=True, error_action="ignore", stepwise=True,
    )
    return float(np.asarray(model.predict(n_periods=horizon))[-1])


def make_fc_sarimax(order=(1, 1, 1), seasonal=(0, 0, 0, 0)):
    def forecast(train_y, train_exog, future_exog, horizon):
        try:
            from statsmodels.tsa.statespace.sarimax import SARIMAX
        except ImportError as exc:
            raise RuntimeError("Thieu statsmodels: pip install statsmodels") from exc
        kwargs = dict(
            order=order, seasonal_order=seasonal,
            enforce_stationarity=False, enforce_invertibility=False,
        )
        if train_exog is not None and train_exog.shape[1] > 0:
            result = SARIMAX(train_y, exog=train_exog, **kwargs).fit(disp=False)
            return float(np.asarray(result.forecast(steps=horizon, exog=future_exog))[-1])
        result = SARIMAX(train_y, **kwargs).fit(disp=False)
        return float(np.asarray(result.forecast(steps=horizon))[-1])
    return forecast


## 3. VECM (đa biến: nội địa ↔ London)

In [ ]:
# ============================================================
# 3) VECM
# ============================================================
def walk_forward_vecm(df, cols=("Gia_target", "london_vnd_kg"), horizon=1, k_ar_diff=1):
    try:
        from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
    except ImportError as exc:
        raise RuntimeError("Thieu statsmodels: pip install statsmodels") from exc

    work = df[[DATE_COL, *cols]].dropna().copy().reset_index(drop=True)
    values = work[list(cols)].astype(float).to_numpy()
    dates = pd.to_datetime(work[DATE_COL]).to_numpy()
    serie = values[:, 0]
    origins = evaluation_origins(len(values), horizon)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(origins, f"VECM h={horizon}"):
        train = values[:d + 1]
        try:
            rank = max(select_coint_rank(train, det_order=0, k_ar_diff=k_ar_diff, signif=0.05).rank, 1)
            fitted = VECM(train, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic="ci").fit()
            prediction = float(fitted.predict(steps=horizon)[-1, 0])
        except Exception:
            prediction = float(serie[d])
        y_pred.append(prediction)
        y_true.append(float(serie[d + horizon]))
        y_naive.append(float(serie[d]))
        base.append(float(serie[d]))
    result = {k: np.asarray(v, float) for k, v in {
        "y_true": y_true, "y_pred": y_pred, "y_naive": y_naive, "base": base
    }.items()}
    result["origins"] = np.asarray(origins, int)
    result["target_dates"] = pd.to_datetime(dates[result["origins"] + horizon])
    return result


## 4. ARIMA-GARCH — coverage khoảng 95%

In [ ]:
# ============================================================
# 4) ARIMA-GARCH COVERAGE
# ============================================================
def garch_coverage(serie, horizon=1):
    try:
        from arch import arch_model
    except ImportError as exc:
        raise RuntimeError("Thieu arch: pip install arch") from exc

    serie = np.asarray(serie, float)
    returns = pd.Series(serie).pct_change().dropna().to_numpy() * 100.0
    n = len(returns)
    start = int(n * INITIAL_FRAC)
    inside = total = 0
    for d in progress(range(start, n - horizon + 1, STEP), f"GARCH h={horizon}"):
        try:
            fitted = arch_model(
                returns[:d], mean="AR", lags=1, vol="GARCH", p=1, q=1, dist="t"
            ).fit(disp="off")
            forecast = fitted.forecast(horizon=horizon, reindex=False)
            mean_steps = forecast.mean.values[-1]
            variance_steps = forecast.variance.values[-1]
            cumulative_mean = float(mean_steps.sum())
            cumulative_std = float(np.sqrt(variance_steps.sum()))
            low = cumulative_mean - 1.96 * cumulative_std
            high = cumulative_mean + 1.96 * cumulative_std
            actual = float((serie[d + horizon - 1] / serie[d - 1] - 1.0) * 100.0)
            inside += int(low <= actual <= high)
            total += 1
        except Exception:
            continue
    return inside / total if total else float("nan")


## 5. Deep Learning — NHITS / NBEATSx

In [ ]:
# ============================================================
# 5) NEURALFORECAST: NHITS / NBEATSx
# ============================================================
def run_dl(df, horizon=63, n_windows=30, max_steps=500, refit=1, save_model=False):
    try:
        import torch
        from neuralforecast import NeuralForecast
        from neuralforecast.models import NHITS, NBEATSx
    except ImportError as exc:
        raise RuntimeError("Thieu neuralforecast/torch: pip install neuralforecast torch") from exc

    n = len(df)
    if horizon <= 0 or horizon >= n:
        raise ValueError("horizon khong hop le")
    long_df = pd.DataFrame({
        "unique_id": "cafe",
        "ds": np.arange(n, dtype=int),
        "y": df[TARGET].astype(float).to_numpy(),
    })
    use_gpu = torch.cuda.is_available()
    accelerator = "gpu" if use_gpu else "cpu"
    devices = 1
    common = dict(
        h=horizon, input_size=max(2 * horizon, 30), max_steps=max_steps,
        scaler_type="robust", accelerator=accelerator, devices=devices,
        enable_progress_bar=False, enable_model_summary=False, random_seed=42,
    )
    models = [
        NHITS(**common, n_blocks=[1, 1, 1], alias="NHITS"),
        NBEATSx(**common, alias="NBEATSx"),
    ]
    nf = NeuralForecast(models=models, freq=1)
    kwargs = dict(df=long_df, n_windows=n_windows, step_size=STEP, verbose=False)
    if refit > 0:
        kwargs["refit"] = refit
    try:
        cv = nf.cross_validation(**kwargs)
    except TypeError:
        kwargs.pop("refit", None)
        cv = nf.cross_validation(**kwargs)

    cv_h = cv.sort_values(["cutoff", "ds"]).groupby("cutoff", sort=False).tail(1).reset_index(drop=True)
    for tag in ("NHITS", "NBEATSx"):
        if tag in cv_h.columns:
            valid = cv_h[tag].notna() & cv_h["y"].notna()
            print(tag, "MAE =", round(mae(cv_h.loc[valid, "y"], cv_h.loc[valid, tag]), 1))

    if save_model:
        final_nf = NeuralForecast(models=models, freq=1)
        final_nf.fit(df=long_df, val_size=min(max(horizon, 1), max(1, n // 10)))
        path = NF_MODEL_DIR / f"nhits_nbeatsx_h{horizon}"
        path.mkdir(parents=True, exist_ok=True)
        final_nf.save(path=str(path), model_index=None, overwrite=True, save_dataset=True)
        print("Saved:", path)
    return cv_h


## 6. Foundation model zero-shot — Chronos

In [ ]:
# ============================================================
# 6) CHRONOS ZERO-SHOT
# ============================================================
def _load_chronos(model_id, device):
    import torch
    from chronos import ChronosPipeline
    dtype = torch.float32 if device == "cpu" else torch.bfloat16
    errors = []
    for kwargs in ({"torch_dtype": dtype}, {"dtype": dtype}, {}):
        try:
            return ChronosPipeline.from_pretrained(model_id, device_map=device, **kwargs)
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
    raise RuntimeError("Khong load duoc Chronos: " + " | ".join(errors))


def _to_numpy(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "float"):
        value = value.float()
    if hasattr(value, "cpu"):
        value = value.cpu()
    return np.asarray(value)


def run_chronos(serie, dates=None, horizon=63, context=512, model_id="amazon/chronos-bolt-base"):
    try:
        import torch
        import chronos  # noqa
    except ImportError as exc:
        raise RuntimeError("Thieu chronos-forecasting: pip install chronos-forecasting torch") from exc

    serie = np.asarray(serie, float)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipeline = _load_chronos(model_id, device)
    origins = evaluation_origins(len(serie), horizon)
    y_true, y_pred, y_naive, base = [], [], [], []
    for d in progress(origins, f"Chronos h={horizon}"):
        ctx = torch.tensor(serie[max(0, d - context + 1):d + 1], dtype=torch.float32)
        forecast = pipeline.predict(context=ctx, prediction_length=horizon)
        arr = _to_numpy(forecast[0])
        median_path = np.quantile(arr, 0.5, axis=0) if arr.ndim >= 2 else arr
        prediction = float(np.asarray(median_path).reshape(-1)[-1])
        y_pred.append(prediction)
        y_true.append(float(serie[d + horizon]))
        y_naive.append(float(serie[d]))
        base.append(float(serie[d]))
    result = {k: np.asarray(v, float) for k, v in {
        "y_true": y_true, "y_pred": y_pred, "y_naive": y_naive, "base": base
    }.items()}
    result["origins"] = np.asarray(origins, int)
    if dates is not None:
        result["target_dates"] = pd.to_datetime(np.asarray(dates)[result["origins"] + horizon])
    return result


## 7. Chạy statistical models và xuất bảng so sánh

In [ ]:
# ============================================================
# 7) RUNNER
# ============================================================
def run_all():
    df = load_data()
    serie = df[TARGET].astype(float).to_numpy()
    dates = pd.to_datetime(df[DATE_COL]).to_numpy()

    exog_cols = [c for c in ["london_vnd_kg_lag1", "usdvnd_lag1", "oni", "rain_90d"] if c in df.columns]
    exog = df[exog_cols].astype(float).ffill().to_numpy() if exog_cols else None
    fc_sarimax = make_fc_sarimax(order=(1, 1, 1))
    london = next(
        (c for c in ["london_vnd_kg", "london_vnd_kg_lag1"] if c in df.columns),
        next((c for c in df.columns if "london" in c.lower()), None),
    )

    # AutoARIMA va VECM co the rat cham. Chuyen False neu chi muon smoke test nhanh.
    USE = {
        "Naive": True,
        "AutoARIMA": True,
        "SARIMA_m5": False,
        "SARIMAX": True,
        "VECM": True,
    }

    jobs = []
    for h in HORIZONS:
        if USE["Naive"]:
            jobs.append(("Naive", h, lambda h=h: walk_forward_eval(serie, fc_naive, h, dates=dates, desc=f"Naive h={h}")))
        if USE["AutoARIMA"]:
            jobs.append(("AutoARIMA", h, lambda h=h: walk_forward_eval(serie, fc_autoarima, h, dates=dates, desc=f"AutoARIMA h={h}")))
        if USE["SARIMA_m5"]:
            jobs.append(("SARIMA_m5", h, lambda h=h: walk_forward_eval(serie, fc_sarima, h, dates=dates, desc=f"SARIMA h={h}")))
        if USE["SARIMAX"]:
            jobs.append(("SARIMAX", h, lambda h=h: walk_forward_eval(serie, fc_sarimax, h, exog=exog, dates=dates, desc=f"SARIMAX h={h}")))
        if USE["VECM"] and london is not None:
            jobs.append(("VECM", h, lambda h=h: walk_forward_vecm(df, cols=(TARGET, london), horizon=h)))

    out_path = TS_DIR / "ts_model_comparison.csv"
    rows = []
    for index, (tag, horizon, fn) in enumerate(jobs, 1):
        started = time.time()
        print(f"[{_now()}] ({index}/{len(jobs)}) {tag} h={horizon} dang chay...")
        try:
            result = fn()
            row = summarize(tag, horizon, result)
            target_dates = result.get("target_dates")
            if target_dates is None:
                target_dates = dates[result["origins"] + horizon]
            save_preds(
                tag, horizon, target_dates,
                result["y_true"], result["y_pred"],
                anchor=result["base"], outdir=PREDS_DIR / "ts",
            )
            rows.append(row)
            pd.DataFrame(rows).to_csv(out_path, index=False, encoding="utf-8-sig")
            print(f"  OK {time.time()-started:.1f}s | MAE={row['MAE']} | DA={row['DA']}")
        except Exception as exc:
            print(f"  LOI {time.time()-started:.1f}s: {type(exc).__name__}: {exc}")

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["h", "MAE"]).reset_index(drop=True)
        out.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(out.to_string(index=False))

    print("\nGARCH coverage 95%:")
    for h in HORIZONS:
        try:
            print(f"h={h}: {garch_coverage(serie, h):.3f}")
        except Exception as exc:
            print(f"h={h}: LOI {type(exc).__name__}: {exc}")
    return out


# Chay dong duoi sau khi da kiem tra DATA_PATH:
# results = run_all()
